In [ ]:
import pydicom
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import medfilt
import os
import matplotlib.patches as patches
from scipy.signal import find_peaks, savgol_filter, correlate

In [ ]:
def get_calibration_from_metadata(ds):
    """
    Searches DICOM metadata for the Spectral Doppler region (RegionDataType == 3)
    to retrieve precise scaling factors.
    """
    delta_x = 0.01 # Default fallback (seconds/pixel)
    delta_y = 0.1  # Default fallback (cm/s/pixel)
    found = False
    if "SequenceOfUltrasoundRegions" in ds:
        for i, region in enumerate(ds.SequenceOfUltrasoundRegions):
            # RegionDataType: 1=2D, 2=M-Mode, 3=Spectral Doppler, 4=Color Flow
            r_type = getattr(region, 'RegionDataType', 0)
            
            # We specifically want Type 3 (Spectral) or if units suggest velocity (3 = cm/sec)
            units_y = getattr(region, 'PhysicalUnitsYDirection', 0)
            
            if r_type == 3 or units_y == 3:
                try:
                    # PhysicalDeltaX = seconds per pixel
                    # PhysicalDeltaY = velocity units per pixel
                    dx = region.PhysicalDeltaX
                    dy = region.PhysicalDeltaY
                    
                    # Sanity check: avoid zero divisions or zero scales
                    if dx > 0 and dy > 0:
                        delta_x = dx
                        delta_y = dy
                        print(f"Metadata Calibration Found (Region {i}): X={delta_x:.5f} s/px, Y={delta_y:.5f} cm/s/px")
                        found = True
                        break
                except AttributeError:
                    continue
    
    if not found:
        print("Warning: Spectral Doppler metadata not found. Using default calibration.")
    return delta_x, delta_y

In [ ]:
def find_baseline_and_crop(pixel_array):
    """
    Locates the orange baseline to define the exact X and Y bounds of the spectral region.
    Uses a 'density march' from the baseline upwards to find the true top of the waveform,
    ignoring floating artifacts like text labels.
    """
    height, width, _ = pixel_array.shape
    
    # 1. Define Search Area (Bottom half only to be safe)
    search_offset_y = height // 2
    search_area = pixel_array[search_offset_y:, :, :]
    
    # 2. Create Mask for Orange/Yellow Baseline
    # R > 130, G between 80-255, B < 100 is a robust yellow/orange filter
    red_mask = search_area[:,:,0] > 130
    green_mask = (search_area[:,:,1] > 80) & (search_area[:,:,1] <= 255)
    blue_mask = search_area[:,:,2] < 100 
    
    orange_mask = red_mask & green_mask & blue_mask
    
    # 3. Detect Baseline Row (Y-Coordinate)
    # ADJUSTED HERE: Improved detection to find the TOP EDGE of the baseline
    row_counts = np.sum(orange_mask, axis=1)
    max_orange_count = np.max(row_counts)
    
    if max_orange_count < 50:
        return None, None, None, None # No valid line found
        
    # Identify all rows that contain a significant portion of the baseline (e.g., > 50% of max width)
    # This helps handle lines that might be 2-4 pixels thick.
    baseline_candidate_rows = np.where(row_counts > (max_orange_count * 0.5))[0]
    
    # Pick the smallest index (the highest row physically) to be the cut-off point.
    # This ensures we crop exactly ABOVE the orange line, excluding the line itself from the ROI.
    relative_baseline_y = np.min(baseline_candidate_rows)

    # Convert to absolute image coordinate
    baseline_y = relative_baseline_y + search_offset_y
    
    # 4. Detect Baseline Width (X-Coordinates)
    # Look at the specific row identified as the baseline (or slightly below it to be safe for width)
    # We use the row with the MAX count to determine width safely
    width_check_row = np.argmax(row_counts)
    baseline_row_pixels = orange_mask[width_check_row, :]
    orange_indices = np.where(baseline_row_pixels)[0]
    
    if len(orange_indices) == 0:
        return None, None, None, None
    min_x = orange_indices[0]
    max_x = orange_indices[-1]
    
    # 5. Detect Top Limit (Ceiling) using Density Marching
    # We start at the baseline and move UP row by row.
    # We count how many "signal" pixels are in each row.
    # If the count drops near zero for several consecutive rows, we stop.
    # This effectively "disconnects" the waveform from the floating text above it.
    
    # Crop relevant area above baseline
    # Look up to 300 pixels high
    look_ahead = 300
    roi_start_y = max(0, relative_baseline_y - look_ahead)
    density_check_region = search_area[roi_start_y:relative_baseline_y, min_x:max_x]
    
    # Convert to grayscale
    if len(density_check_region.shape) == 3:
        d_gray = np.mean(density_check_region, axis=2)
    else:
        d_gray = density_check_region
        
    # Strong threshold to see only the bright white signal
    signal_mask = d_gray > 40 
    
    # Iterate from BOTTOM (baseline) to TOP
    # 'rows' here are 0 at top, so we iterate backward from height-1 to 0
    h_roi, w_roi = d_gray.shape
    
    # We need a gap of N empty rows to declare "end of waveform"
    gap_tolerance = 10 
    current_gap = 0
    ceiling_index = 0 # Default to top of ROI if no gap found
    
    # Start from bottom (which is the baseline) and go up
    for r in range(h_roi - 1, -1, -1):
        # Count bright pixels in this row
        pixel_count = np.sum(signal_mask[r, :])
        
        # If row is empty (or just speckle noise < 5 pixels)
        if pixel_count < 5:
            current_gap += 1
        else:
            # Reset gap counter if we hit signal again
            current_gap = 0
            
        # If we found a large enough gap, this is the ceiling
        if current_gap > gap_tolerance:
            # The ceiling is the row index + gap size (going downwards)
            ceiling_index = r + gap_tolerance
            break
            
    # Convert relative ceiling index back to absolute Y coordinate
    # relative_baseline_y is the bottom of this ROI
    # The ROI starts at roi_start_y.
    # The ceiling index is relative to roi_start_y.
    
    relative_top_y = roi_start_y + ceiling_index
    
    # Add a small safety buffer (10px) above the detected signal top
    relative_top_y = max(0, relative_top_y - 10)
    
    # Convert to absolute image coordinates
    top_y = search_offset_y + relative_top_y
    
    return min_x, max_x, top_y, baseline_y

In [ ]:
def extract_traces(roi, delta_x, delta_y):
    """
    Analyzes the ROI column by column.
    """
    height, width = roi.shape
    
    upper_line = []
    lower_line = []
    mean_velocity = []
    time_axis = []
    
    # Threshold to separate signal (white/grey) from background (black)
    threshold = 25 
    
    for col in range(width):
        column_data = roi[:, col]
        
        # Flip so index 0 is the baseline (0 velocity)
        column_flip = np.flip(column_data)
        
        # Find indices of signal pixels
        signal_indices = np.where(column_flip > threshold)[0]
        
        if len(signal_indices) > 3: 
            # 1. Upper Line (Highest velocity)
            # 98th percentile is robust against single salt-and-pepper noise pixels
            max_px = np.percentile(signal_indices, 98)
            
            # 2. Lower Line (Lowest velocity near baseline)
            # FIXED: Used 5th percentile instead of min() to ignore bottom clutter/noise
            # FIXED: Removed the check "if min_px < 5: min_px = 0" which was forcing the line flat
            min_px = np.percentile(signal_indices, 5)

            # 3. Mean Velocity (Weighted Average)
            intensities = column_flip[signal_indices]
            weighted_avg_px = np.average(signal_indices, weights=intensities)
            
            # Convert to physical units
            # delta_y is usually cm/s/px. Divide by 100 for m/s.
            v_upper = (max_px * delta_y) / 100.0
            v_lower = (min_px * delta_y) / 100.0
            v_mean  = (weighted_avg_px * delta_y) / 100.0
            
        else:
            v_upper = 0.0
            v_lower = 0.0
            v_mean  = 0.0
            
        upper_line.append(v_upper)
        lower_line.append(v_lower)
        mean_velocity.append(v_mean)
        time_axis.append(col * delta_x)
        
    return time_axis, upper_line, lower_line, mean_velocity

In [ ]:
def shift_cycle_start(peak_incdies, velocity_array):
    
    new_cycle_points = []
    previous_end_point = 0
    next_peaks = np.delete(peak_incdies, 0)
    next_peaks = np.append(next_peaks, np.shape(velocity_array)[0])

    for peak,next_peak in zip(peak_incdies, next_peaks):
        thr_up = np.percentile(velocity_array[peak-20:next_peak-30], 95)
        idx_start = np.where(velocity_array[previous_end_point:peak] >= thr_up)

        thr_dwn = np.percentile(velocity_array[peak:next_peak-20], 9)
        idx_end = np.where(velocity_array[peak:next_peak-1] < thr_dwn)
        new_cycle_points.append([idx_start[0][0] + previous_end_point, idx_end[0][-1] + peak])
        previous_end_point = new_cycle_points[-1][1]

    return new_cycle_points





In [ ]:
def represnetative_cycle(velo_array):
        # 1. Load your data
    # --------------------------------------------------------------
    # Replace this line with your own way of loading data
    # Example: velocity = np.loadtxt("velocity.csv")
    velocity = np.array(velo_array)     # already extracted from the image

    # --------------------------------------------------------------
    # 2. (Optional) Smooth the signal
    # --------------------------------------------------------------

    velocity = savgol_filter(velocity, window_length=31, polyorder=3)


    # --------------------------------------------------------------
    # 3. Detect systolic peaks to define cycles
    # --------------------------------------------------------------
    # Minimum distance between peaks (adjust based on sampling rate)


    height_peaks_upper = np.percentile(velocity, 85)

    # peaks_upper, _ = find_peaks(upper, distance=75,height=height_peaks_upper)

    peak_indices, _ = find_peaks(
        velocity, 
        distance=75,
        height=height_peaks_upper,
        #prominence=5    # adjust if peaks are too small/large
    )

    print("Detected peaks at indices:", peak_indices)

    peak_indices = shift_cycle_start(peak_indices, velocity)

    print("Detected peaks at indices:", peak_indices)

    # --------------------------------------------------------------
    # 4. Extract cycles between peaks
    # --------------------------------------------------------------
    cycles = []
    for start, end in peak_indices:
        print(start,end)

        cycles.append(velocity[start:end])


    print("Shifting complete.")
    # --------------------------------------------------------------
    # 5. Align cycles via cross-correlation
    # --------------------------------------------------------------
    aligned_cycles = []
    ref_cycle = cycles[0]   # first cycle is reference

    for cyc in cycles:
        # full cross-correlation
        corr = correlate(cyc, ref_cycle, mode='full')
        lag = np.argmax(corr) - (len(ref_cycle) - 1)
        aligned = np.roll(cyc, lag)
        aligned_cycles.append(aligned)

    # --------------------------------------------------------------
    # 6. Resample every cycle to same length
    # --------------------------------------------------------------
    target_len = min(len(c) for c in aligned_cycles)  # or choose a fixed number
    resampled_cycles = []

    for cyc in aligned_cycles:
        old_t = np.linspace(0, 1, len(cyc))
        new_t = np.linspace(0, 1, target_len)
        cyc_resampled = np.interp(new_t, old_t, cyc)
        resampled_cycles.append(cyc_resampled)

    resampled_cycles = np.array(resampled_cycles)

    # --------------------------------------------------------------
    # 7. Compute representative (mean) cycle
    # --------------------------------------------------------------
    avg_cycle = np.mean(resampled_cycles, axis=0)

    return avg_cycle, velocity, peak_indices, resampled_cycles

In [ ]:
def visualize_merging_process(avg_cycle, velocity, peak_indices, resampled_cycles):

    peak_indices = np.array(peak_indices) #fixes issues with indicies during plotting

    plt.figure(figsize=(14, 8))

    plt.subplot(3, 1, 1)
    plt.title("Original Velocity Signal with Detected Peaks")
    plt.plot(velocity, label="Signal")


    plt.plot(
    peak_indices[:, 0],
    velocity[peak_indices[:, 0]],
    "ro",
    label="Start point"
    )

    # Second column: black points labeled as End point
    plt.plot(
        peak_indices[:, 1],
        velocity[peak_indices[:, 1]],
        "ko",
        label="End point"
    )
    plt.legend()

    # Individual cycles (aligned)
    plt.subplot(3, 1, 2)
    plt.title("Aligned Cycles")
    for cyc in resampled_cycles:
        plt.plot(cyc, alpha=0.4)
    plt.legend(["Cycles"])

    # Averaged cycle
    plt.subplot(3, 1, 3)
    plt.title("Representative Averaged Cycle")
    plt.plot(avg_cycle, linewidth=3, color="black")
    plt.ylim(bottom = 0.0)
    plt.legend(["Averaged Cycle"])

    plt.tight_layout()
    plt.show()


In [ ]:
def show_avereage_representative_cycle(avg_cycle):
    # Averaged cycle
    plt.subplot(3, 1, 3)
    plt.title("Representative Averaged Cycle")
    plt.plot(avg_cycle, linewidth=3, color="black")
    plt.ylim(bottom = 0.0)
    plt.legend(["Averaged Cycle"])

    plt.tight_layout()
    plt.show()


In [ ]:

# --- 1. CONFIGURATION ---
file_path = "path/to/doppler_dicom_file"  # set to your own spectral Doppler DICOM

# --- 2. LOAD DATA ---
ds = pydicom.dcmread(file_path)
pixel_data = ds.pixel_array

print(pixel_data.shape)

# --- 3. GET CALIBRATION ---
delta_x, delta_y = get_calibration_from_metadata(ds)

# --- 4. DETECT REGION ---
min_x, max_x, top_y, baseline_y = find_baseline_and_crop(pixel_data)


if baseline_y is None:
    print("Error: Baseline not found.")
print(f"Detected Region: X[{min_x}:{max_x}], Y[{top_y}:{baseline_y}]")

# --- 5. PREPARE ROI ---
# Crop exactly to the detected bounds
roi_color = pixel_data[top_y:baseline_y, min_x:max_x]

if len(roi_color.shape) == 3:
    roi_gray = np.mean(roi_color, axis=2)
else:
    roi_gray = roi_color

# --- 6. EXTRACT TRACES ---
t, upper, lower, mean = extract_traces(roi_gray, delta_x, delta_y)
# --------------------------------------------------------------

representative_cycle_upper_velocities, velocity_array, peak_indices, resampled_cycles = represnetative_cycle(mean)

visualize_merging_process(representative_cycle_upper_velocities, velocity_array, peak_indices, resampled_cycles)

# print(peak_indices, len(resampled_cycles))

# for cycle in resampled_cycles:
#     print(len(cycle))



# My stuff in case of some future adjustments


representative_cycle_lower_velocities, velocity_array, peak_indices, resampled_cycles = lower_represnetative_cycle(lower)

visualize_merging_process(representative_cycle_lower_velocities, velocity_array, peak_indices, resampled_cycles)

avg_cycle = np.mean(representative_cycle_lower_velocities,representative_cycle_upper_velocities)

show_avereage_representative_cycle(avg_cycle)
